# Native radio-access-network antenna placement
Regenerate all thirteen original seed0 coefficients. Retain at most seven selected antennas, all neighbor equality groups, three modeled binary slack bits, four QAOA layers, penalty3, sixty mean-cost iterations and3000actual final CPU shots. Preserve the original maximization sign. Native exact means and public Nelder-Mead replace private shot-based optimization; each vertex is synthesized. Saved Qmod coefficient tables are not a fresh-data substitute.

## Objective sign and modeled slack are explicit semantics

The visible qfunc prepares thirteen location bits and three binary slack bits,
then composes all four source cost/mixer layers. FlagQuantum compiles fresh
real coefficients, phases and named outputs. The source maximizes its positive
consumption score; the migrated model deliberately preserves that sign rather
than silently turning it into a consumption-minimization problem.

Try another eight-angle preview binding and inspect group equalities and the
seven-location limit separately from score. The sixty-iteration host search
uses shared quantum.py and exact local means, followed by 3000 actual draws.
Propagate model edits there. Slack outputs are part of the search domain,
not temporary workspace, and finite observations do not certify an optimum.


In [ ]:
from flagquantum import synth
from flagquantum.synth import Output
from flagquantum.synth import QArray
from flagquantum.synth import QBit
from flagquantum.synth import RX
from flagquantum.synth import allocate
from flagquantum.synth import hadamard_transform
from flagquantum.synth import phase
from flagquantum.synth import qfunc
import math

from examples.synthesis.applications.telecom.radio_access_network.radio_access_network_positioning_antennas.overview.quantum import (
    validate_coefficients,
)


@qfunc
def native_main(
    params: list[float],
    coefficients: list[float],
    x: Output[QArray[QBit]],
    slack: Output[QArray[QBit]],
) -> None:
    allocate(13, x)
    allocate(3, slack)
    hadamard_transform(x)
    hadamard_transform(slack)
    for layer in range(4):
        for i in range(13):
            phase(coefficients[i] * x[i], params[layer])
        phase(-3 * (x[1] + x[3] + x[4] - 1) ** 2, params[layer])
        phase(-3 * (x[0] + x[5] - 1) ** 2, params[layer])
        phase(-3 * (x[6] + x[9] - 1) ** 2, params[layer])
        phase(
            -3
            * (
                x[0]
                + x[1]
                + x[2]
                + x[3]
                + x[4]
                + x[5]
                + x[6]
                + x[7]
                + x[8]
                + x[9]
                + x[10]
                + x[11]
                + x[12]
                + slack[0]
                + 2 * slack[1]
                + 4 * slack[2]
                - 7
            )
            ** 2,
            params[layer],
        )
        for i in range(13):
            RX(params[4 + layer], x[i])
        for i in range(3):
            RX(params[4 + layer], slack[i])


def build_preview(
    parameters: tuple[float, ...], coefficients: tuple[float, ...]
) -> synth.Program:
    """Bind source four-layer angles and freshly generated classical data."""
    validate_coefficients(coefficients)
    if len(parameters) != 8 or any(
        type(value) not in (int, float) or not math.isfinite(value)
        for value in parameters
    ):
        raise ValueError("four layers require eight finite real angles")
    return synth.synthesize(
        native_main,
        classical_inputs={"params": parameters, "coefficients": coefficients},
        max_qubits=16,
        max_workspace=0,
        max_operations=10000,
        memory_limit_bytes=32 * 1024**2,
    )


import json
import random
from pathlib import Path

from IPython.display import Image, display

from examples.synthesis.applications.telecom.radio_access_network.radio_access_network_positioning_antennas.overview.model import (
    GROUPS,
    fresh_coefficients,
)
from examples.synthesis.applications.telecom.radio_access_network.radio_access_network_positioning_antennas.overview.plots import (
    plot_results,
)
from examples.synthesis.applications.telecom.radio_access_network.radio_access_network_positioning_antennas.overview.quantum import (
    build_program,
)
from examples.synthesis.applications.telecom.radio_access_network.radio_access_network_positioning_antennas.overview.workflow import (
    INITIAL_PARAMETERS,
    run_workflow,
)
from flagquantum.drawer import draw

random.seed(0)
coefficients = fresh_coefficients()
print("Fresh source seed0 coefficients:", coefficients)
print("Thirteen potential locations; at most seven selected antennas.")


In [ ]:
print("Original neighbor equality groups:", GROUPS)
print("Each group has exactly one selected antenna.")
print("The source MAXIMIZES the positive consumption score; this sign is retained.")
print("Three modeled slack bits have binary weights1,2,4.")


In [ ]:
print("Four source QAOA layers; penalty factor3.")
print("All16bits are mixed; model slack is separate from temporary workspace.")


In [ ]:
initial_program = build_preview(INITIAL_PARAMETERS, coefficients)
print("Fresh real coefficients and angles bound at synthesis time.")
print("Native synthesis objective:", initial_program.selection.objective)
print("No saved Hamiltonian coefficient table or runtime-binding substitution.")


In [ ]:
shots = 3000
print("Actual final CPU sampling shots:", shots)
print("Exact local mean replaces shot estimates during optimization.")


In [ ]:
print(initial_program.report["resources"])
print(draw(initial_program.circuit, show_all_qubits=True))


In [ ]:
workflow = run_workflow(maxiter=60, shots=shots, seed=2718)
assert workflow.coefficients == coefficients
print("Actual objective evaluations:", len(workflow.trace))
print("Optimizer iterations:", workflow.optimizer.iterations)


In [ ]:
plot_paths = plot_results(workflow, Path("artifacts/development/radio-access-network"))
display(Image(filename=str(plot_paths[0])))


In [ ]:
print("Top five actual joint observations:")
for row in workflow.samples[:5]:
    print(row, "source penalized cost=", row.cost)
best = workflow.samples[0]
print("Selected locations:", best.assignment.selected)
print("Actual maximized score:", best.assignment.score)
print("Decision feasibility:", best.assignment.feasible)
print("Residual penalty:", best.penalty)
assert sum(row.count for row in workflow.samples) == 3000


In [ ]:
display(Image(filename=str(plot_paths[1])))
report = {
    "decision_bits": 13,
    "slack_bits": 3,
    "width": 16,
    "layers": 4,
    "maxiter": 60,
    "iterations": workflow.optimizer.iterations,
    "evaluations": len(workflow.trace),
    "quantile": 1,
    "shots": sum(row.count for row in workflow.samples),
    "coefficients": list(coefficients),
    "best_word": best.assignment.word,
    "best_slack": best.slack,
    "best_cost": best.cost,
    "best_score": best.assignment.score,
    "best_penalty": best.penalty,
    "best_feasible": best.assignment.feasible,
    "trace": [value.cost for value in workflow.trace],
}
print("ARTIFACT_JSON=" + json.dumps(report))
